# 02 — Cross-Venue CEX Diagnostics

**Purpose:** Merge Bybit + Binance futures onto the master calendar. Analyze spreads and validate market integration (justifies using Bybit as primary venue).

**Inputs:** master calendar + Bybit klines/OI/funding + Binance futures
**Outputs:**
- `data/analysis/datasets/cex_diagnostics_1h.parquet`
- `data/analysis/reports/diagnostics_qa.json`

This is a diagnostic notebook, not the baseline econometric dataset.

In [5]:
# ── Setup ──
import sys; sys.path.insert(0, "..")
from config import CFG, REPORTS_DIR
CFG.ensure_dirs()

import pandas as pd
import numpy as np
import json

print(f"Project root: {CFG.ROOT}")

Project root: .


In [7]:
# ── 1. Load master calendar ──

cal = pd.read_parquet(CFG.FILES.master_calendar, engine="pyarrow")
cal["date"] = pd.to_datetime(cal["date"], utc=True)
print(f"Calendar: {len(cal):,} hours")

Calendar: 41,328 hours


In [9]:
# ── 2. Load & standardize each source ──

def load_parquet(path, cols_rename=None):
    """Load parquet, ensure UTC date, optionally rename columns."""
    df = pd.read_parquet(path)
    df["date"] = pd.to_datetime(df["date"], utc=True)
    if cols_rename:
        df = df.rename(columns=cols_rename)
    return df

# Bybit klines
bybit_k = load_parquet(CFG.FILES.bybit_klines)
bybit_k = bybit_k[["date", "open", "high", "low", "close", "volume"]].copy()
bybit_k.columns = ["date"] + [f"{c}_bybit" for c in bybit_k.columns[1:]]

# Bybit OI
bybit_oi = load_parquet(CFG.FILES.bybit_oi)
oi_col = [c for c in bybit_oi.columns if c != "date"][0]  # handles varying col names
bybit_oi = bybit_oi[["date", oi_col]].rename(columns={oi_col: "oi_bybit"})

# Bybit funding
bybit_f = load_parquet(CFG.FILES.bybit_funding)
fund_col = [c for c in bybit_f.columns if c != "date"][0]
bybit_f = bybit_f[["date", fund_col]].rename(columns={fund_col: "funding_bybit"})

# Binance futures
binance = load_parquet(CFG.FILES.binance_futures)
binance = binance[["date", "close"]].rename(columns={"close": "close_binance"})

print("All sources loaded:")
for name, df in [("bybit_klines", bybit_k), ("bybit_oi", bybit_oi),
                  ("bybit_funding", bybit_f), ("binance", binance)]:
    print(f"  {name:18s}: {len(df):,} rows, [{df['date'].min()}, {df['date'].max()}]")

All sources loaded:
  bybit_klines      : 41,328 rows, [2021-03-15 00:00:00+00:00, 2025-11-30 23:00:00+00:00]
  bybit_oi          : 41,328 rows, [2021-03-15 00:00:00+00:00, 2025-11-30 23:00:00+00:00]
  bybit_funding     : 41,328 rows, [2021-03-15 00:00:00+00:00, 2025-11-30 23:00:00+00:00]
  binance           : 43,080 rows, [2021-01-01 00:00:00+00:00, 2025-11-30 23:00:00+00:00]


In [11]:
# ── 3. Left-join everything onto calendar ──

panel = cal.copy()
for df in [bybit_k, bybit_oi, bybit_f, binance]:
    panel = panel.merge(df, on="date", how="left")

# Derived columns
panel["price_spread_bps"] = 1e4 * (panel["close_bybit"] - panel["close_binance"]) / panel["close_binance"]
panel["ret_bybit"]  = np.log(panel["close_bybit"]).diff()
panel["ret_binance"] = np.log(panel["close_binance"]).diff()

print(f"Panel shape: {panel.shape}")
print(f"\nMissing values:")
for c in ["close_bybit", "close_binance", "oi_bybit", "funding_bybit"]:
    n_miss = panel[c].isna().sum()
    print(f"  {c:20s}: {n_miss:,}  ({100*n_miss/len(panel):.2f}%)")

Panel shape: (41328, 12)

Missing values:
  close_bybit         : 0  (0.00%)
  close_binance       : 0  (0.00%)
  oi_bybit            : 0  (0.00%)
  funding_bybit       : 0  (0.00%)


In [13]:
# ── 4. Spread diagnostics ──

spread = panel["price_spread_bps"].dropna()

print("═" * 55)
print("PRICE SPREAD: BYBIT vs BINANCE (bps)")
print("═" * 55)
print(f"Mean (bias)        : {spread.mean():.4f} bps")
print(f"Std (volatility)   : {spread.std():.4f} bps")
print(f"Mean |spread|      : {spread.abs().mean():.4f} bps")
print(f"Median |spread|    : {spread.abs().median():.4f} bps")
print(f"P99 |spread|       : {np.percentile(spread.abs(), 99):.2f} bps")
print(f"P99.9 |spread|     : {np.percentile(spread.abs(), 99.9):.2f} bps")

corr = panel[["ret_bybit", "ret_binance"]].dropna().corr().iloc[0, 1]
print(f"\nReturn correlation : {corr:.6f}")
print("→ Confirms near-perfect market integration" if corr > 0.999 else "→ Check for micro-structure differences")

═══════════════════════════════════════════════════════
PRICE SPREAD: BYBIT vs BINANCE (bps)
═══════════════════════════════════════════════════════
Mean (bias)        : -0.3373 bps
Std (volatility)   : 3.2671 bps
Mean |spread|      : 1.9775 bps
Median |spread|    : 1.3442 bps
P99 |spread|       : 9.42 bps
P99.9 |spread|     : 14.17 bps

Return correlation : 0.998950
→ Check for micro-structure differences


In [15]:
# ── 5. Save ──

panel.to_parquet(CFG.FILES.cex_diagnostics, index=False, engine="pyarrow")
print(f"Saved: {CFG.FILES.cex_diagnostics}")

qa = {
    "panel_rows": len(panel),
    "missing_bybit_close": int(panel["close_bybit"].isna().sum()),
    "missing_binance_close": int(panel["close_binance"].isna().sum()),
    "spread_mean_bps": float(spread.mean()),
    "spread_std_bps": float(spread.std()),
    "return_correlation": float(corr),
    "status": "PASS" if corr > 0.99 else "CHECK",
}

from config import REPORTS_DIR
qa_path = REPORTS_DIR / "diagnostics_qa.json"
with open(qa_path, "w") as f:
    json.dump(qa, f, indent=2)
print(f"Saved: {qa_path}")

print("\n✅ Notebook 02 complete")

Saved: ./data/analysis/datasets/cex_diagnostics_1h.parquet
Saved: ./data/analysis/reports/diagnostics_qa.json

✅ Notebook 02 complete
